# Direct base-model tools — no adapter
Choose **GPU T4 x2**, enable **Internet**, and attach the private dataset made from **tool_use_source.zip**. Set MODEL_SIZE below. No old contract adapter, selective-facts run, GitHub push or earlier result bundle is required.

Both arms use NF4, the same prompt, JSON tool protocol, bounded tool loop and open-development tasks. Calls are chosen by the model, not automatically inserted. This is custom JSON tool use, not native function calling.

This notebook evaluates the **intent parser**, not the six end-to-end Stage 2 systems. It scores 80 synthetic and 20 existing development tasks and records actual tool calls, correct-reference acceptance and wrong-output acceptance. These are development diagnostics, not an independent test benchmark.

The training notebook first runs a two-step smoke job, then fresh two-epoch QLoRA training. It evaluates the unadapted base and both epoch checkpoints and selects the best candidate under the documented semantic screening rule. A completed training job does not guarantee an eligible adapter. Only the adapter evaluation notebook requires the NEW tool adapter dataset.

Download the output ZIP and executed notebook. Do not run both sizes in the same output directory or attach an old contract/selective-facts adapter here.


In [1]:
MODEL_SIZE = "1_5b"  # choose "1_5b" or "3b"; use the same size for a fair comparison
assert MODEL_SIZE in ("1_5b", "3b")


In [2]:
import hashlib,json,os,subprocess,sys,zipfile
from pathlib import Path
WORK=Path('/kaggle/working')
WORK.mkdir(parents=True,exist_ok=True)
MANIFEST_SHA='4d0f5ffa0f7464c17451f3d5e5112fd0d6433a5b3e22070e543d9aace4ce8d03'
INPUT=Path('/kaggle/input')
MANIFEST_NAME='tool_use_manifest.json'
ROOT=WORK/('tool_use_source_'+MANIFEST_SHA[:12])

def write_source(manifest_bytes,read_source):
    assert hashlib.sha256(manifest_bytes).hexdigest()==MANIFEST_SHA, 'Wrong source/data bundle version'
    manifest=json.loads(manifest_bytes)
    assert manifest['experiment']=='tool-use-v2'
    for name,expected in manifest['sha256'].items():
        relative=Path(name)
        assert not relative.is_absolute() and '..' not in relative.parts, 'Invalid source path'
        content=read_source(name)
        assert hashlib.sha256(content).hexdigest()==expected, f'Changed source/data: {name}'
        path=ROOT/relative;path.parent.mkdir(parents=True,exist_ok=True)
        if path.exists():assert path.read_bytes()==content, f'Changed working copy: {name}'
        else:path.write_bytes(content)
    print('Verified source and data:',len(manifest['sha256']),'files',MANIFEST_SHA)

# Kaggle can expose uploaded ZIPs as files or unpacked dataset directories. Support both.
manifest_paths=[p for p in INPUT.rglob(MANIFEST_NAME) if hashlib.sha256(p.read_bytes()).hexdigest()==MANIFEST_SHA]
if manifest_paths:
    manifest_path=manifest_paths[0]
    write_source(manifest_path.read_bytes(),lambda name:(manifest_path.parent/name).read_bytes())
else:
    archives=list(INPUT.rglob('tool_use_source.zip'))
    found=False
    for archive in archives:
        with zipfile.ZipFile(archive) as z:
            assert len(z.namelist())==len(set(z.namelist())), 'Duplicate source ZIP members'
            name='tool_use_source/'+MANIFEST_NAME
            if name not in z.namelist():continue
            manifest_bytes=z.read(name)
            if hashlib.sha256(manifest_bytes).hexdigest()!=MANIFEST_SHA:continue
            write_source(manifest_bytes,lambda relative:z.read('tool_use_source/'+relative))
            found=True;break
    assert found, 'Attach the private dataset made from tool_use_source.zip. No original adapter is required.'
PACKAGES=WORK/'tool_use_packages'
subprocess.run([sys.executable,'-m','pip','install','-q','--upgrade','--target',str(PACKAGES),'-r',str(ROOT/'requirements.txt')],check=True)
ENV=dict(os.environ,PYTHONPATH=os.pathsep.join(map(str,[ROOT,ROOT/'src',PACKAGES])),PYTHONUNBUFFERED='1',TOKENIZERS_PARALLELISM='false')


Verified source and data: 169 files 4d0f5ffa0f7464c17451f3d5e5112fd0d6433a5b3e22070e543d9aace4ce8d03


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.51.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
kaggle-environments 1.33.3 requires PyYAML>=6.0.3, but you have pyyaml 6.0.2 which is incompatible.
ydata-profiling 4.18.4 requires PyYAML<6.1,>=6.0.3, but you have pyyaml 6.0.2 which is incompatible.
aiobotocore 2.26.0 requires python-dateutil<3.0.0,>=2.1, but you have python-dateutil 3.0.0 which is incompatible.
botocore 1.41.5 requires python-dateutil<3.0.0,>=2.1, but you have python-dateutil 3.0.0 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.20.0, but you have jupyter-server 2.12.5 which is incompatible.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 2.3.3 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
d

In [3]:
subprocess.run([sys.executable,'-m','pip','install','-q','transformers==5.20.2','peft==0.21.2','bitsandbytes==0.50.2','accelerate==1.15.0','tokenizers==0.23.3','safetensors==0.8.0','huggingface-hub==1.33.0'],check=True)
subprocess.run([sys.executable,'-m','eval.stage2_independent_v2.environment','--prepare'],cwd=ROOT,env=ENV,check=True)
subprocess.run([sys.executable,'-m','finetune.tool_use.preflight','--size',MODEL_SIZE],cwd=ROOT,env=ENV,check=True)
subprocess.run([sys.executable,'-c',"import torch; assert torch.__version__=='2.11.0+cu130'; assert torch.cuda.device_count()==2; assert all('T4' in torch.cuda.get_device_name(i) for i in (0,1)); print('T4 x2 preflight passed')"],cwd=ROOT,env=ENV,check=True)
for physical_gpu in ['0','1']:
    subprocess.run([sys.executable,'-m','finetune.selective_facts.attention'],cwd=ROOT,env=dict(ENV,CUDA_VISIBLE_DEVICES=physical_gpu),check=True)


Removing optional torchao 0.10.0: incompatible with PEFT 0.21.2; this workflow does not use torchao.
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
{"torchao": null, "action": "removed_incompatible", "previous_version": "0.10.0"}
{"probe": "cpu_lora_injection_and_forward", "trainable_parameters": 16, "status": "passed"}
TrainingArguments API passed; warmup_steps=0.03 (3% ratio)


100%|██████████| 1/1 [00:00<00:00,  3.06it/s]
                                             
100%|██████████| 1/1 [00:00<00:00,  2.94it/s]


{'loss': '3.494', 'grad_norm': '0.0003399', 'learning_rate': '0', 'epoch': '1'}
{'eval_loss': '3.494', 'eval_runtime': '0.007', 'eval_samples_per_second': '142.3', 'eval_steps_per_second': '142.3', 'epoch': '1'}
{'train_runtime': '0.3397', 'train_samples_per_second': '2.944', 'train_steps_per_second': '2.944', 'train_loss': '3.494', 'epoch': '1'}
PASSED: tiny CPU Trainer, assistant labels, LoRA optimization, adapter save/reload


PASSED actual tokenizer: no-tool, tool, recovery and escalation conversations; every assistant turn supervised. Lengths: [2635, 2252, 2709, 2223]
T4 x2 preflight passed
{"event": "attention_visible_devices", "devices": [0], "cuda_visible_devices": "0"}
{"event": "attention_shape", "device": "cuda:0", "query_shape": [1, 4, 32, 128], "original_key_shape": [1, 2, 32, 128], "expanded_key_shape": [1, 4, 32, 128], "dtype": "torch.float16", "native_gqa_expanded": true, "stage": "prefill", "has_mask": false, "is_causal": true, "backend": "EFFICIENT_ATTENTION", "math_fallback_allowed": false}
{"event": "kernel_self_test_passed", "device": "cuda:0", "case": "causal_prefill", "max_absolute_error": 0.001953125, "rtol": 0.005, "atol": 0.005}
{"event": "attention_shape", "device": "cuda:0", "query_shape": [1, 4, 1, 128], "original_key_shape": [1, 2, 32, 128], "expanded_key_shape": [1, 4, 32, 128], "dtype": "torch.float16", "native_gqa_expanded": true, "stage": "decode", "has_mask": false, "is_causal

In [4]:
def execute(smoke):
    out=WORK/('tool_use_base_'+MODEL_SIZE+('_smoke' if smoke else ''))
    assert not out.exists(), 'Use a fresh session: existing results will not be overwritten.'
    command=[sys.executable,'-m','finetune.tool_use.evaluate','--size',MODEL_SIZE,'--out',str(out),'--packages',str(PACKAGES)]
    if smoke:command+=['--smoke']
    log=WORK/(out.name+'.log')
    with log.open('w') as f:
        process=subprocess.Popen(command,cwd=ROOT,env=dict(ENV,CUDA_VISIBLE_DEVICES='0'),stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        for line in process.stdout:
            print(line,end='');f.write(line);f.flush()
        code=process.wait()
    assert code==0, f'Job failed. Download {log} and the partial ZIP for diagnosis.'
    return out
SMOKE=execute(True)
print('Smoke passed. The next cell starts a fresh full job.')



Loading weights: 100%|██████████| 338/338 [00:02<00:00, 165.49it/s]
DEV 1/2 warehouse_fill_0: ok
DEV 2/2 warehouse_dedup_0: ok
Smoke passed. The next cell starts a fresh full job.


In [5]:
OUT=execute(False)
from IPython.display import display,FileLink
display(FileLink(str(OUT.with_suffix('.zip'))))
print((OUT/'summary.json').read_text())



Loading weights: 100%|██████████| 338/338 [00:01<00:00, 170.12it/s]
DEV 1/100 warehouse_filter_0: parser_invalid_response
DEV 2/100 warehouse_filter_1: parser_invalid_response
DEV 3/100 warehouse_filter_2: parser_invalid_response
DEV 4/100 warehouse_filter_3: parser_invalid_response
DEV 5/100 warehouse_dedup_0: ok
DEV 6/100 warehouse_dedup_1: ok
DEV 7/100 warehouse_dedup_2: ok
DEV 8/100 warehouse_dedup_3: ok
DEV 9/100 warehouse_fill_0: ok
DEV 10/100 warehouse_fill_1: ok
DEV 11/100 warehouse_fill_2: parser_invalid_response
DEV 12/100 warehouse_fill_3: parser_invalid_response
DEV 13/100 warehouse_convert_0: ok
DEV 14/100 warehouse_convert_1: ok
DEV 15/100 warehouse_convert_2: ok
DEV 16/100 warehouse_convert_3: ok
DEV 17/100 warehouse_edit_0: ok
DEV 18/100 warehouse_edit_1: ok
DEV 19/100 warehouse_edit_2: ok
DEV 20/100 warehouse_edit_3: parser_token_limit
DEV 21/100 solar_filter_0: parser_invalid_response
DEV 22/100 solar_filter_1: parser_invalid_response
DEV 23/100 solar_filter_2: parse

/kaggle/working/tool_use_base_1_5b.zip

{
  "complete": true,
  "smoke": false,
  "summary": {
    "synthetic": {
      "tasks": 80,
      "correct_accepted": 13,
      "wrong_accepted": 21,
      "joint_success": 9,
      "invalid": 33,
      "tokens": 418053,
      "tasks_with_tools": 8,
      "tool_attempts": 34,
      "tool_failures": 5
    },
    "original": {
      "tasks": 20,
      "correct_accepted": 1,
      "wrong_accepted": 1,
      "joint_success": 1,
      "invalid": 18,
      "tokens": 199391,
      "tasks_with_tools": 8,
      "tool_attempts": 22,
      "tool_failures": 10
    }
  },
  "provenance": {
    "runtime": {
      "eval/stage2_tools/__init__.py": "7dd68d5f2df342915b366c7744797fd23cedbc99b29c15b2feacc6b46033b682",
      "eval/stage2_tools/backend.py": "df9c4cb6131e0292ec3955b50d8f2b32025622564e3bd3c10af7ab356f777459",
      "eval/stage2_tools/collect.py": "8ce89c99afa40377265ad707ba79b7ab4903e03880e2b0ea4722d26ad09d3e08",
      "eval/stage2_tools/config.py": "a0264dbe35a82143e68e4148c1d5a44905c167896